In [ ]:
#Cell 1
# Colab: install pinned versions for reproducibility
!pip install -q "transformers==4.44.2" "rouge-score==0.1.2" "sentencepiece==0.2.0" "torch>=2.2.0"

import os, random, numpy as np, torch, pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.4 MB/s eta 0:00:00


device(type='cpu')

In [ ]:
#Cell 2
from google.colab import files
uploaded = files.upload()  # upload meeting_minutes_cleaned.csv

df = pd.read_csv("meeting_minutes_cleaned.csv")
assert {"transcript","summary"}.issubset(df.columns), "CSV must have transcript & summary columns"
print(df.shape, df.columns.tolist())
df.head(2)

Saving meeting_minutes_cleaned.csv to meeting_minutes_cleaned.csv
(214, 2) ['transcript', 'summary']


,transcript,summary
0,project manager introduced upcoming project te...,project manager introduced upcoming project te...
1,project manager briefed team new requirement c...,project manager briefed team new requirement c...


In [ ]:
#Cell 3
MODEL_NAME = "facebook/bart-large-cnn"

def load_bart(model_name=MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    model.eval()
    return tokenizer, model

tokenizer, model = load_bart()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
#Cell 4
@torch.inference_mode()
def generate_batch_summaries(
    texts,
    tokenizer,
    model,
    max_input_len=1024,
    max_sum_len=120,
    min_sum_len=40,
    num_beams=5,
    length_penalty=1.1,
    no_repeat_ngram_size=3,
    batch_size=8
):
    outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_input_len, return_tensors="pt"
        ).to(device)

        gen_ids = model.generate(
            **enc,
            max_length=max_sum_len,
            min_length=min_sum_len,
            num_beams=num_beams,
            length_penalty=length_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
            early_stopping=True
        )
        outs = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
        outputs.extend(outs)
    return outputs

def chunk_text(text, tokenizer, max_tokens=900):
    toks = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for i in range(0, len(toks), max_tokens):
        sub = toks[i:i+max_tokens]
        chunks.append(tokenizer.decode(sub, skip_special_tokens=True))
    return chunks

def map_reduce_summarize(
    text,
    tokenizer,
    model,
    map_max_input=900,
    map_max_sum=96,
    reduce_max_input=1024,
    reduce_max_sum=128
):
    pieces = chunk_text(text, tokenizer, max_tokens=map_max_input)
    if len(pieces) <= 1:
        return generate_batch_summaries(
            [text], tokenizer, model,
            max_input_len=reduce_max_input, max_sum_len=reduce_max_sum,
            min_sum_len=40, num_beams=5, length_penalty=1.1, no_repeat_ngram_size=3,
            batch_size=1
        )[0]

    # Map: summarize each chunk
    chunk_summaries = generate_batch_summaries(
        pieces, tokenizer, model,
        max_input_len=map_max_input, max_sum_len=map_max_sum,
        min_sum_len=24, num_beams=4, length_penalty=1.0, no_repeat_ngram_size=3,
        batch_size=4
    )

    # Reduce: summarize concatenation of chunk summaries
    stitched = " ".join(chunk_summaries)
    final = generate_batch_summaries(
        [stitched], tokenizer, model,
        max_input_len=reduce_max_input, max_sum_len=reduce_max_sum,
        min_sum_len=40, num_beams=5, length_penalty=1.1, no_repeat_ngram_size=3,
        batch_size=1
    )[0]
    return final


In [ ]:
#Cell 5
def run_bart_on_dataframe(df: pd.DataFrame, sample_n=10, hierarchical=True):
    data = df.copy()
    if sample_n is not None:
        data = data.head(sample_n).copy()

    preds = []
    for txt in tqdm(data["transcript"].tolist(), desc="Summarizing with BART"):
        if hierarchical:
            pred = map_reduce_summarize(txt, tokenizer, model)
        else:
            pred = generate_batch_summaries(
                [txt], tokenizer, model,
                max_input_len=1024, max_sum_len=120, min_sum_len=40,
                num_beams=5, length_penalty=1.1, no_repeat_ngram_size=3,
                batch_size=1
            )[0]
        preds.append(pred)

    out_df = data.copy()
    out_df["generated_summary"] = preds
    return out_df

# Change sample_n to None to run full dataset
bart_out = run_bart_on_dataframe(df, sample_n=10, hierarchical=True)
bart_out[["transcript","summary","generated_summary"]].head(3)


Summarizing with BART: 100%|██████████| 10/10 [02:33<00:00, 15.32s/it]


,transcript,summary,generated_summary
0,project manager introduced upcoming project te...,project manager introduced upcoming project te...,project manager introduced upcoming project te...
1,project manager briefed team new requirement c...,project manager briefed team new requirement c...,project manager briefed team new requirement c...
2,project manager recapped decision made previou...,project manager recapped decision made previou...,project manager recapped decision made previou...


In [ ]:
#Cell 6
def compute_rouge(df_eval: pd.DataFrame):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL', 'rougeLsum'], use_stemmer=True
    )
    r1 = r2 = rl = rls = 0.0
    rows = 0
    for ref, hyp in zip(df_eval["summary"], df_eval["generated_summary"]):
        if not isinstance(ref, str) or not isinstance(hyp, str):
            continue
        s = scorer.score(ref, hyp)
        r1 += s['rouge1'].fmeasure
        r2 += s['rouge2'].fmeasure
        rl += s['rougeL'].fmeasure
        rls += s['rougeLsum'].fmeasure
        rows += 1
    return {
        "ROUGE-1": r1/max(rows,1),
        "ROUGE-2": r2/max(rows,1),
        "ROUGE-L": rl/max(rows,1),
        "ROUGE-Lsum": rls/max(rows,1),
        "N": rows
    }

metrics = compute_rouge(bart_out)
metrics


{'ROUGE-1': 0.4614536496748327,
 'ROUGE-2': 0.4533095942846379,
 'ROUGE-L': 0.4614536496748327,
 'ROUGE-Lsum': 0.4614536496748327,
 'N': 10}

In [ ]:
#Cell 7
bart_out.to_csv("bart_summarized_output.csv", index=False)
from google.colab import files
files.download("bart_summarized_output.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>